In [ ]:
from pathlib import Path
from datetime import datetime
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# --- Rutas base del proyecto ---
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "dset-s2"
WEIGHTS_DIR = PROJECT_DIR / "weights"
EXPERIMENTS_DIR = PROJECT_DIR / "experiments"

cfg = {
    "train_img_dir": DATA_DIR / "tra_scene",
    "train_mask_dir": DATA_DIR / "tra_truth",
    "val_img_dir": DATA_DIR / "val_scene",
    "val_mask_dir": DATA_DIR / "val_truth",
    
    # Parámetros del parche y bandas (B, G, R, NIR)
    "patch_size": 256,
    "train_stride": 256,
    "val_stride": 256,
    "band_ids": [1, 2, 3, 4],
    "architecture": "Unet",
    "encoder_name": "resnet34",
    
    # Hiperparámetros de entrenamiento
    "batch_size": 8,
    "epochs": 40,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "num_workers": 0,
    "min_water_ratio": 0.001,
    "empty_patch_keep_prob": 0.20,
    
    # Early stopping y Scheduler (ReduceLROnPlateau)
    "early_stopping_patience": 6,
    "early_stopping_min_delta": 1e-4,
    "lr_patience": 2,
    "lr_factor": 0.5,
    "seed": 42,
    
    # Tracking de experimentos
    "experiment_name": f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
}

cfg["experiment_dir"] = EXPERIMENTS_DIR / cfg["experiment_name"]
cfg["model_path"] = cfg["experiment_dir"] / "best_model.pth"
cfg["history_path"] = cfg["experiment_dir"] / "history.csv"
cfg["config_path"] = cfg["experiment_dir"] / "config.json"
cfg["val_metrics_path"] = cfg["experiment_dir"] / "val_metrics.csv"
cfg["learning_curve_path"] = cfg["experiment_dir"] / "learning_curves.png"
cfg["confusion_matrix_path"] = cfg["experiment_dir"] / "confusion_matrix.png"
cfg["pr_curve_path"] = cfg["experiment_dir"] / "precision_recall_curve.png"
cfg["experiment_dir"].mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)


def build_run_summary(cfg):
    return (
        f"{cfg['architecture']} + {cfg['encoder_name']} | "
        f"patch={cfg['patch_size']} | batch={cfg['batch_size']} | "
        f"lr={cfg['lr']} | wd={cfg['weight_decay']} | "
        f"train_stride={cfg['train_stride']} | val_stride={cfg['val_stride']}"
    )


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [13]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def mask_name_from_img(img_path):
    return img_path.name.replace("_6Bands_", "_").replace(".tif", "_Truth.tif")


def collect_pairs(img_dir, mask_dir):
    pairs = []
    for img_path in sorted(img_dir.glob("*.tif")):
        mask_path = mask_dir / mask_name_from_img(img_path)
        if mask_path.exists():
            pairs.append((img_path, mask_path))
    return pairs


def build_positions(size, patch_size, stride):
    if size <= patch_size:
        return [0]

    pos = list(range(0, size - patch_size + 1, stride))
    last = size - patch_size
    if pos[-1] != last:
        pos.append(last)
    return pos


def build_samples(
    pairs,
    patch_size,
    stride,
    filter_empty=False,
    min_water_ratio=0.001,
    empty_patch_keep_prob=0.20,
):
    samples = []
    for img_path, mask_path in pairs:
        with rasterio.open(img_path) as src:
            rows = build_positions(src.height, patch_size, stride)
            cols = build_positions(src.width, patch_size, stride)

        if not filter_empty:
            for row in rows:
                for col in cols:
                    samples.append((img_path, mask_path, row, col))
            continue

        with rasterio.open(mask_path) as src:
            for row in rows:
                for col in cols:
                    window = Window(col_off=col, row_off=row, width=patch_size, height=patch_size)
                    mask = src.read(1, window=window)
                    water_ratio = float((mask > 0).mean())

                    # Guardamos todos los parches con agua y solo una parte de los vacios.
                    if water_ratio >= min_water_ratio or random.random() < empty_patch_keep_prob:
                        samples.append((img_path, mask_path, row, col))
    return samples


set_seed(cfg["seed"])


In [14]:
class WaterDataset(Dataset):
    def __init__(self, samples, patch_size=256, band_ids=None, is_train=False):
        self.samples = samples
        self.patch_size = patch_size
        self.band_ids = band_ids or [1, 2, 3, 4]
        self.is_train = is_train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, row, col = self.samples[idx]
        window = Window(col_off=col, row_off=row, width=self.patch_size, height=self.patch_size)

        with rasterio.open(img_path) as src:
            img = src.read(self.band_ids, window=window).astype(np.float32)

        with rasterio.open(mask_path) as src:
            mask = src.read(1, window=window).astype(np.float32)

        # Sentinel-2 suele venir escalado en reflectancia * 10000.
        img = img / 10000.0

        blue = img[0]
        green = img[1]
        red = img[2]
        nir = img[3]

        # NDWI ayuda a resaltar agua frente a vegetacion y suelo.
        ndwi = (green - nir) / (green + nir + 1e-6)

        img = np.stack([blue, green, red, nir, ndwi], axis=0)

        # La mascara tiene que quedar en [1, H, W] para BCE y Dice.
        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)

        if self.is_train:
            # Un poco de augmentacion ayuda a no memorizar tanto los parches.
            if random.random() < 0.5:
                img = np.flip(img, axis=2).copy()
                mask = np.flip(mask, axis=2).copy()

            if random.random() < 0.5:
                img = np.flip(img, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()

            if random.random() < 0.5:
                k = random.randint(1, 3)
                img = np.rot90(img, k=k, axes=(1, 2)).copy()
                mask = np.rot90(mask, k=k, axes=(1, 2)).copy()

            if random.random() < 0.5:
                scale = random.uniform(0.90, 1.10)
                img[:4] = np.clip(img[:4] * scale, 0.0, 1.0)
                img[4] = (img[1] - img[3]) / (img[1] + img[3] + 1e-6)

            if random.random() < 0.3:
                noise = np.random.normal(0.0, 0.01, size=img[:4].shape).astype(np.float32)
                img[:4] = np.clip(img[:4] + noise, 0.0, 1.0)
                img[4] = (img[1] - img[3]) / (img[1] + img[3] + 1e-6)

        return torch.from_numpy(img), torch.from_numpy(mask)


In [15]:
train_pairs = collect_pairs(cfg["train_img_dir"], cfg["train_mask_dir"])
val_pairs = collect_pairs(cfg["val_img_dir"], cfg["val_mask_dir"])

train_samples = build_samples(
    train_pairs,
    cfg["patch_size"],
    cfg["train_stride"],
    filter_empty=True,
    min_water_ratio=cfg["min_water_ratio"],
    empty_patch_keep_prob=cfg["empty_patch_keep_prob"],
)
val_samples = build_samples(val_pairs, cfg["patch_size"], cfg["val_stride"])

print("train scenes:", len(train_pairs))
print("val scenes:", len(val_pairs))
print("train patches:", len(train_samples))
print("val patches:", len(val_samples))

water_pixels = 0
total_pixels = 0
for _, mask_path, row, col in train_samples:
    window = Window(col_off=col, row_off=row, width=cfg["patch_size"], height=cfg["patch_size"])
    with rasterio.open(mask_path) as src:
        mask = src.read(1, window=window)
    water_pixels += int((mask > 0).sum())
    total_pixels += int(mask.size)

water_ratio = water_pixels / max(total_pixels, 1)
neg_pixels = total_pixels - water_pixels
cfg["pos_weight"] = float(neg_pixels / max(water_pixels, 1))

print("train water ratio:", round(water_ratio, 6))
print("bce pos_weight:", round(cfg["pos_weight"], 4))

train_ds = WaterDataset(train_samples, patch_size=cfg["patch_size"], band_ids=cfg["band_ids"], is_train=True)
val_ds = WaterDataset(val_samples, patch_size=cfg["patch_size"], band_ids=cfg["band_ids"], is_train=False)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg["batch_size"],
    shuffle=True,
    num_workers=cfg["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg["batch_size"],
    shuffle=False,
    num_workers=cfg["num_workers"],
    pin_memory=torch.cuda.is_available(),
)


train scenes: 64
val scenes: 31
train patches: 3115
val patches: 522


KeyboardInterrupt: 

In [ ]:
model = smp.Unet(
    encoder_name=cfg["encoder_name"],
    encoder_weights=None,
    in_channels=5,
    classes=1,
)
model = model.to(device)

pos_weight = torch.tensor([cfg["pos_weight"]], dtype=torch.float32, device=device)
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)

# weight_decay mete una penalizacion suave para no sobreajustar tanto.
optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=cfg["lr_factor"],
    patience=cfg["lr_patience"],
)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())


def calc_loss(logits, mask):
    bce = bce_loss(logits, mask)
    dice = dice_loss(logits, mask)
    loss = 0.5 * bce + 0.5 * dice
    return loss, bce, dice


def calc_metrics(logits, mask, thr=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    pred = (probs > thr).float()

    pred = pred.view(pred.size(0), -1)
    mask = mask.view(mask.size(0), -1)

    inter = (pred * mask).sum(dim=1)
    union = pred.sum(dim=1) + mask.sum(dim=1) - inter
    dice = (2 * inter + eps) / (pred.sum(dim=1) + mask.sum(dim=1) + eps)
    iou = (inter + eps) / (union + eps)

    return dice.mean().item(), iou.mean().item()


In [ ]:
def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_bce = 0.0
    total_dice_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    pbar = tqdm(loader, leave=False)
    for img, mask in pbar:
        img = img.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(img)
                loss, bce, dice_part = calc_loss(logits, mask)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

        dice_score, iou_score = calc_metrics(logits.detach(), mask.detach())

        total_loss += loss.item()
        total_bce += bce.item()
        total_dice_loss += dice_part.item()
        total_dice += dice_score
        total_iou += iou_score

        pbar.set_description(f"loss={loss.item():.4f} dice={dice_score:.4f} iou={iou_score:.4f}")

    n = len(loader)
    return {
        "loss": total_loss / n,
        "bce": total_bce / n,
        "dice_loss": total_dice_loss / n,
        "dice": total_dice / n,
        "iou": total_iou / n,
    }


In [ ]:
history = []
best_iou = -1.0
epochs_without_improving = 0

for epoch in range(1, cfg["epochs"] + 1):
    train_metrics = run_epoch(train_loader, model, optimizer=optimizer)
    with torch.no_grad():
        val_metrics = run_epoch(val_loader, model, optimizer=None)

    scheduler.step(val_metrics["iou"])

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_bce": train_metrics["bce"],
        "train_dice_loss": train_metrics["dice_loss"],
        "train_dice": train_metrics["dice"],
        "train_iou": train_metrics["iou"],
        "val_loss": val_metrics["loss"],
        "val_bce": val_metrics["bce"],
        "val_dice_loss": val_metrics["dice_loss"],
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
        "lr": optimizer.param_groups[0]["lr"],
        "epochs_without_improving": epochs_without_improving,
    }
    history.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} train_dice={row['train_dice']:.4f} train_iou={row['train_iou']:.4f} | "
        f"val_loss={row['val_loss']:.4f} val_dice={row['val_dice']:.4f} val_iou={row['val_iou']:.4f}"
    )

    if row["val_iou"] > best_iou + cfg["early_stopping_min_delta"]:
        best_iou = row["val_iou"]
        epochs_without_improving = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "cfg": cfg,
                "epoch": epoch,
                "val_iou": row["val_iou"],
                "val_dice": row["val_dice"],
            },
            cfg["model_path"],
        )
        print(f"mejor modelo guardado en {cfg['model_path']} con val_iou={best_iou:.4f}")
    else:
        epochs_without_improving += 1
        print(f"sin mejora clara en val_iou por {epochs_without_improving} epocas")

    if epochs_without_improving >= cfg["early_stopping_patience"]:
        print(f"early stopping en epoch {epoch:02d} por falta de mejora en validacion")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(cfg["history_path"], index=False)

cfg_to_save = {}
for key, value in cfg.items():
    if isinstance(value, Path):
        cfg_to_save[key] = str(value)
    else:
        cfg_to_save[key] = value

with open(cfg["config_path"], "w", encoding="utf-8") as f:
    json.dump(cfg_to_save, f, indent=2, ensure_ascii=False)

print("experimento guardado en:", cfg["experiment_dir"])
print("history:", cfg["history_path"])
print("config:", cfg["config_path"])
print("best model:", cfg["model_path"])

history_df.tail()


In [ ]:
def load_model_for_inference(model_path, device):
    # El checkpoint es local y lo generamos nosotros en entrenamiento.
    ckpt = torch.load(model_path, map_location=device, weights_only=False)

    model_cfg = ckpt.get("cfg", {})
    band_ids = model_cfg.get("band_ids", cfg["band_ids"])

    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=5,
        classes=1,
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device)
    model.eval()

    return model, band_ids, model_cfg


def preprocess_patch(img_patch):
    img_patch = img_patch.astype(np.float32) / 10000.0

    blue = img_patch[0]
    green = img_patch[1]
    red = img_patch[2]
    nir = img_patch[3]

    ndwi = (green - nir) / (green + nir + 1e-6)
    img_patch = np.stack([blue, green, red, nir, ndwi], axis=0)

    return img_patch


def predict_tif(
    img_path,
    model,
    device,
    band_ids,
    patch_size=256,
    stride=256,
    threshold=0.5,
    out_path=None,
):
    img_path = Path(img_path)

    with rasterio.open(img_path) as src:
        height, width = src.height, src.width
        rows = build_positions(height, patch_size, stride)
        cols = build_positions(width, patch_size, stride)

        prob_sum = np.zeros((height, width), dtype=np.float32)
        count_sum = np.zeros((height, width), dtype=np.float32)
        meta = src.meta.copy()

        with torch.no_grad():
            for row in tqdm(rows, desc=f"infer {img_path.name}", leave=False):
                for col in cols:
                    window = Window(col_off=col, row_off=row, width=patch_size, height=patch_size)
                    img_patch = src.read(band_ids, window=window)
                    img_patch = preprocess_patch(img_patch)

                    x = torch.from_numpy(img_patch).unsqueeze(0).to(device)
                    with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                        logits = model(x)
                        probs = torch.sigmoid(logits).squeeze().detach().cpu().numpy()

                    row_end = row + patch_size
                    col_end = col + patch_size
                    prob_sum[row:row_end, col:col_end] += probs
                    count_sum[row:row_end, col:col_end] += 1.0

    prob_map = prob_sum / np.clip(count_sum, a_min=1e-6, a_max=None)
    pred_mask = (prob_map >= threshold).astype(np.uint8)

    if out_path is not None:
        out_path = Path(out_path)
        meta.update(count=1, dtype="uint8")
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(pred_mask, 1)

    return prob_map, pred_mask


In [ ]:
def plot_learning_curves(history_df, cfg):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    axes[0].set_title("Loss Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_iou"], label="train_iou")
    axes[1].plot(history_df["epoch"], history_df["val_iou"], label="val_iou")
    axes[1].set_title("IoU Curve")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("IoU")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.suptitle("Learning Curves", fontsize=14, y=1.04)
    fig.text(0.5, 0.98, build_run_summary(cfg), ha="center", va="top", fontsize=9)
    plt.tight_layout()
    fig.savefig(cfg["learning_curve_path"], dpi=200, bbox_inches="tight")
    plt.show()


def evaluate_classification_metrics(loader, model, threshold=0.5):
    model.eval()

    all_probs = []
    all_mask = []

    with torch.no_grad():
        for img, mask in tqdm(loader, leave=False):
            img = img.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(img)
                probs = torch.sigmoid(logits)

            all_probs.append(probs.detach().cpu().numpy().reshape(-1))
            all_mask.append(mask.numpy().reshape(-1))

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_mask).astype(np.uint8)
    y_pred = (y_prob >= threshold).astype(np.uint8)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    dice = (2 * tp) / max(2 * tp + fp + fn, 1)

    thresholds = np.linspace(0.0, 1.0, 101)
    pr_points = []
    for thr in thresholds:
        pred_thr = (y_prob >= thr).astype(np.uint8)
        tp_thr = ((pred_thr == 1) & (y_true == 1)).sum()
        fp_thr = ((pred_thr == 1) & (y_true == 0)).sum()
        fn_thr = ((pred_thr == 0) & (y_true == 1)).sum()

        precision_thr = tp_thr / max(tp_thr + fp_thr, 1)
        recall_thr = tp_thr / max(tp_thr + fn_thr, 1)
        pr_points.append((recall_thr, precision_thr))

    pr_points = np.array(pr_points)
    order = np.argsort(pr_points[:, 0])
    recall_curve = pr_points[order, 0]
    precision_curve = pr_points[order, 1]
    pr_auc = np.trapezoid(precision_curve, recall_curve)

    metrics = {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "iou": iou,
        "dice": dice,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "pr_auc": pr_auc,
    }

    curves = {
        "recall": recall_curve,
        "precision": precision_curve,
    }

    return metrics, curves


def plot_confusion_matrix(metrics, cfg):
    cm = np.array([
        [metrics["tn"], metrics["fp"]],
        [metrics["fn"], metrics["tp"]],
    ], dtype=np.int64)

    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title("Matriz de Confusion")
    ax.set_xlabel("Prediccion")
    ax.set_ylabel("Real")
    ax.set_xticks([0, 1], labels=["No agua", "Agua"])
    ax.set_yticks([0, 1], labels=["No agua", "Agua"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black")

    fig.suptitle("Validacion | Clasificacion por pixel", fontsize=14, y=1.02)
    fig.text(0.5, 0.96, build_run_summary(cfg), ha="center", va="top", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(cfg["confusion_matrix_path"], dpi=200, bbox_inches="tight")
    plt.show()


def plot_precision_recall_curve(curves, metrics, cfg):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(curves["recall"], curves["precision"], linewidth=2)
    ax.set_title(f"Precision-Recall Curve | AUC={metrics['pr_auc']:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.grid(True, alpha=0.3)
    fig.text(0.5, 0.98, build_run_summary(cfg), ha="center", va="top", fontsize=9)
    plt.tight_layout()
    fig.savefig(cfg["pr_curve_path"], dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
plot_learning_curves(history_df, cfg)

best_model_eval, _, _ = load_model_for_inference(cfg["model_path"], device)
val_metrics_cls, val_curves = evaluate_classification_metrics(val_loader, best_model_eval, threshold=0.5)

metrics_df = pd.DataFrame([val_metrics_cls])
metrics_df.to_csv(cfg["val_metrics_path"], index=False)
print("val metrics:", cfg["val_metrics_path"])
print("learning curves:", cfg["learning_curve_path"])
print("confusion matrix:", cfg["confusion_matrix_path"])
print("pr curve:", cfg["pr_curve_path"])
metrics_df[["threshold", "precision", "recall", "iou", "dice", "pr_auc", "tp", "tn", "fp", "fn"]]

plot_confusion_matrix(val_metrics_cls, cfg)
plot_precision_recall_curve(val_curves, val_metrics_cls, cfg)


In [ ]:
model_inf, band_ids_inf, cfg_inf = load_model_for_inference(cfg["model_path"], device)

# Cambia esta ruta por cualquier Sentinel-2 .tif con las bandas esperadas.
img_path = DATA_DIR / "val_scene" / "S2A_L2A_20190318_N0211_R061_6Bands_S1.tif"
out_path = cfg["experiment_dir"] / "pred_mask_example.tif"

prob_map, pred_mask = predict_tif(
    img_path=img_path,
    model=model_inf,
    device=device,
    band_ids=band_ids_inf,
    patch_size=cfg_inf.get("patch_size", 256),
    stride=cfg_inf.get("patch_size", 256),
    threshold=0.5,
    out_path=out_path,
)

print("imagen:", img_path)
print("salida:", out_path)
print("shape prob:", prob_map.shape)
print("pixeles agua:", int(pred_mask.sum()))


imagen: data\dset-s2\val_scene\S2A_L2A_20190318_N0211_R061_6Bands_S1.tif
salida: pred_mask_example.tif
shape prob: (697, 754)
pixeles agua: 66995
